In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score

In [2]:
X_train = np.load("C:/DS_mainproject/portfolio_prjct/PORTFOLIO_PROJECT/data/processed/X_train.npy")
y_train = np.load("C:/DS_mainproject/portfolio_prjct/PORTFOLIO_PROJECT/data/processed/y_train.npy")
X_test = np.load("C:/DS_mainproject/portfolio_prjct/PORTFOLIO_PROJECT/data/processed/X_test.npy")
y_test = np.load("C:/DS_mainproject/portfolio_prjct/PORTFOLIO_PROJECT/data/processed/y_test.npy")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)


X_train shape: (30976, 30, 15)
y_train shape: (30976,)


In [3]:
lstm_model = Sequential()
lstm_model.add(LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False))
lstm_model.add(Dropout(0.2))
lstm_model.add(Dense(1, activation='sigmoid'))  # Binary classification

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

lstm_model.summary() 

C:\DS_mainproject\portfolio_prjct\portfolio_env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 64)                  │          20,480 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 20,545 (80.25 KB)

 Trainable params: 20,545 (80.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# gru_model = Sequential()
# gru_model.add(GRU(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False))
# gru_model.add(Dropout(0.2))
# gru_model.add(Dense(1, activation='sigmoid'))

# gru_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# gru_model.summary()

In [4]:
from tensorflow.keras.callbacks import EarlyStopping

# Stop training if validation loss does not improve
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# --------- Train LSTM ---------
history_lstm = lstm_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 18s 37ms/step - accuracy: 0.5757 - loss: 0.6736 - val_accuracy: 0.6222 - val_loss: 0.6435
Epoch 2/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.6353 - loss: 0.6404 - val_accuracy: 0.6349 - val_loss: 0.6393
Epoch 3/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.6399 - loss: 0.6377 - val_accuracy: 0.6364 - val_loss: 0.6375
Epoch 4/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.6404 - loss: 0.6351 - val_accuracy: 0.6393 - val_loss: 0.6342
Epoch 5/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.6418 - loss: 0.6329 - val_accuracy: 0.6394 - val_loss: 0.6335
Epoch 6/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.6452 - loss: 0.6321 - val_accuracy: 0.6425 - val_loss: 0.6318
Epoch 7/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.6451 - loss: 0.6310 - val_accuracy: 0.6380 - val_loss: 0.6322
Epoch 8/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.6444 - loss: 0.6289 - val_a

In [5]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Get predicted probabilities
y_pred_prob = lstm_model.predict(X_test)

# Convert probabilities to 0 or 1
y_pred = (y_pred_prob > 0.5).astype(int)

# Flatten arrays (important)
y_test_flat = y_test.reshape(-1)
y_pred_flat = y_pred.reshape(-1)
accuracy = accuracy_score(y_test_flat, y_pred_flat)
precision = precision_score(y_test_flat, y_pred_flat)
recall = recall_score(y_test_flat, y_pred_flat)
f1 = f1_score(y_test_flat, y_pred_flat)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Accuracy : 0.6734245867768595
Precision: 0.6929655762240753
Recall   : 0.7478080295339179
F1 Score : 0.7193430251914327


In [6]:
lstm_model.evaluate(X_test,y_test)

242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6734 - loss: 0.6043


[0.6043494343757629, 0.6734246015548706]

In [7]:
cm = confusion_matrix(y_test_flat, y_pred_flat)
print(cm)

[[1974 1436]
 [1093 3241]]


In [8]:
# Save the entire model (architecture + weights + optimizer state)
lstm_model.save('C:/DS_mainproject/portfolio_prjct/PORTFOLIO_PROJECT/models/lstm_model.h5')

# Or save in the newer Keras format
lstm_model.save('C:/DS_mainproject/portfolio_prjct/PORTFOLIO_PROJECT/models/lstm_model.keras')

print("Model saved successfully!")

Model saved successfully!


In [10]:
# Create models directory if it doesn't exist
import os
models_dir = 'C:/DS_mainproject/portfolio_prjct/PORTFOLIO_PROJECT/models'
os.makedirs(models_dir, exist_ok=True)

# --------- Build GRU Model (same structure as LSTM) ---------
gru_model = Sequential()
gru_model.add(GRU(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False))
gru_model.add(Dropout(0.2))
gru_model.add(Dense(1, activation='sigmoid'))

gru_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\n" + "="*50)
print("GRU MODEL SUMMARY")
print("="*50)
gru_model.summary()


GRU MODEL SUMMARY


C:\DS_mainproject\portfolio_prjct\portfolio_env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ gru (GRU)                            │ (None, 64)                  │          15,552 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 15,617 (61.00 KB)

 Trainable params: 15,617 (61.00 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Early stopping callback
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train GRU
print("\n" + "="*50)
print("TRAINING GRU MODEL")
print("="*50)
history_gru = gru_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# Save GRU model
gru_model.save(os.path.join(models_dir, 'gru_model.keras'))
print("\n✅ GRU model saved successfully!")


TRAINING GRU MODEL
Epoch 1/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.6088 - loss: 0.6561 - val_accuracy: 0.6428 - val_loss: 0.6370
Epoch 2/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.6416 - loss: 0.6365 - val_accuracy: 0.6382 - val_loss: 0.6372
Epoch 3/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.6423 - loss: 0.6333 - val_accuracy: 0.6403 - val_loss: 0.6321
Epoch 4/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.6446 - loss: 0.6313 - val_accuracy: 0.6390 - val_loss: 0.6312
Epoch 5/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.6444 - loss: 0.6289 - val_accuracy: 0.6478 - val_loss: 0.6269
Epoch 6/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.6477 - loss: 0.6251 - val_accuracy: 0.6509 - val_loss: 0.6214
Epoch 7/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.6504 - loss: 0.6217 - val_accuracy: 0.6549 - val_loss: 0.6196
Epoch 8/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.6529 -

In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd

# Function to evaluate model
def evaluate_model(model, X_test, y_test, model_name):
    # Get predictions
    y_pred_prob = model.predict(X_test)
    y_pred = (y_pred_prob > 0.5).astype(int)
    
    # Flatten arrays
    y_test_flat = y_test.reshape(-1)
    y_pred_flat = y_pred.reshape(-1)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test_flat, y_pred_flat)
    precision = precision_score(y_test_flat, y_pred_flat)
    recall = recall_score(y_test_flat, y_pred_flat)
    f1 = f1_score(y_test_flat, y_pred_flat)
    cm = confusion_matrix(y_test_flat, y_pred_flat)
    
    # Print results
    print(f"\n{'='*50}")
    print(f"{model_name} EVALUATION RESULTS")
    print(f"{'='*50}")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"\nConfusion Matrix:")
    print(cm)
    
    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'Confusion Matrix': cm
    }

# Evaluate both models
lstm_results = evaluate_model(lstm_model, X_test, y_test, "LSTM")
gru_results = evaluate_model(gru_model, X_test, y_test, "GRU")

242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step

LSTM EVALUATION RESULTS
Accuracy : 0.6734
Precision: 0.6930
Recall   : 0.7478
F1 Score : 0.7193

Confusion Matrix:
[[1974 1436]
 [1093 3241]]
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step

GRU EVALUATION RESULTS
Accuracy : 0.6839
Precision: 0.6988
Recall   : 0.7649
F1 Score : 0.7303

Confusion Matrix:
[[1981 1429]
 [1019 3315]]


In [13]:
# Create comparison dataframe
comparison_df = pd.DataFrame([
    lstm_results,
    gru_results
])

# Drop confusion matrix from display (too large)
display_df = comparison_df.drop('Confusion Matrix', axis=1)
print("\n" + "="*50)
print("LSTM vs GRU COMPARISON")
print("="*50)
print(display_df.to_string(index=False))

# Find the better model
better_model = display_df.loc[display_df['F1 Score'].idxmax(), 'Model']
better_f1 = display_df['F1 Score'].max()
print(f"\n✅ Better model: {better_model} (F1 Score: {better_f1:.4f})")


LSTM vs GRU COMPARISON
Model  Accuracy  Precision   Recall  F1 Score
 LSTM  0.673425   0.692966 0.747808  0.719343
  GRU  0.683884   0.698777 0.764882  0.730337

✅ Better model: GRU (F1 Score: 0.7303)


In [14]:
# Save the best performing model
if lstm_results['F1 Score'] > gru_results['F1 Score']:
    best_model = lstm_model
    best_model_name = "LSTM"
    best_model.save(os.path.join(models_dir, 'best_model_lstm.keras'))
else:
    best_model = gru_model
    best_model_name = "GRU"
    best_model.save(os.path.join(models_dir, 'best_model_gru.keras'))

print(f"\n✅ Best model ({best_model_name}) saved as 'best_model.keras'")

# Also save comparison results to CSV
comparison_df.to_csv(os.path.join(models_dir, 'model_comparison.csv'), index=False)
print("✅ Model comparison saved to 'model_comparison.csv'")


✅ Best model (GRU) saved as 'best_model.keras'
✅ Model comparison saved to 'model_comparison.csv'
